[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dgunning/edgartools/blob/main/notebooks/revenue-segment-hierarchy-python.ipynb)

# Parse Hierarchical Revenue Segments from SEC XBRL Filings with Python

Use **edgartools** to extract revenue segment hierarchies from SEC filings -- see how Tesla nests sub-segments under parent segments, how Apple breaks revenue into Products vs Services, how Microsoft splits by business segment, and how to build parent-child trees from XBRL data.

**New in v5.21.1:** `to_dataframe(view="detailed")` now preserves **sub-member hierarchy** from the definition linkbase. Sub-members (like Tesla's "Automotive sales" and "Automotive regulatory credits") are properly nested under their parent members (like "Automotive Revenues"), reflected in the `level` column.

**What you'll learn:**
- Extract revenue segments with parent-child hierarchy
- Use `level`, `parent_concept`, and `parent_abstract_concept` columns
- Control detail level with `view` parameter (standard, detailed, summary)
- Build indented tree visualizations from the `level` column
- Build custom segment trees for analysis
- Understand when to use `Statement.to_dataframe()` vs `xbrl.query()`

## Install edgartools

In [1]:
!pip install -q -U edgartools


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## Setup

The SEC requires all automated tools to identify themselves. Replace the email below with your own.

In [2]:
from edgar import *

set_identity("your.name@example.com")

## The Problem: Flat Facts vs Hierarchical Statements

SEC filings report revenue broken down by product, service, geography, and business segment. This data is structured as a **tree** in the XBRL presentation linkbase.

If you use `xbrl.query()` to extract facts, you get a flat list with no hierarchy. The `Statement` object preserves the tree structure -- and `to_dataframe()` gives you columns to navigate it.

## Example 1: Apple Revenue Hierarchy

Apple breaks revenue into Products (iPhone, Mac, iPad, Wearables) and Services. Let's extract this tree.

In [3]:
company = Company("AAPL")
filing = company.get_filings(form="10-K").latest()
xbrl = filing.xbrl()

# Get the income statement -- hierarchy is preserved automatically
income = xbrl.statements.income_statement()
income

                                                                                                     
                                  APPLE INC.   AAPL                                                  
                                  CONSOLIDATED STATEMENT OF INCOME                                   
                                  Sep 30, 2023 to Sep 27, 2025                                       
                                                                                                     
                                                       Sep 27, 2025   Sep 28, 2024   Sep 30, 2023    
   ───────────────────────────────────────────────────────────────────────────────────────────────   
            Net sales:                                     $416,161       $391,035       $383,285    
            Products                                       $307,003       $294,866       $298,085    
            Services                                       $109,158        $96,169

## The Hierarchy Columns

Every `to_dataframe()` call returns these hierarchy columns:

| Column | What it tells you |
|--------|-------------------|
| `level` | Nesting depth (0=root, 1=section, 2=line item, 3=sub-item) |
| `abstract` | True for section headers (no data), False for actual values |
| `parent_concept` | Calculation parent -- the metric this rolls up to for math |
| `parent_abstract_concept` | Presentation parent -- the section header this sits under |

In [4]:
df = income.to_dataframe()

# Show hierarchy columns for the first 15 rows
hierarchy_cols = ['label', 'level', 'abstract', 'parent_concept', 'parent_abstract_concept']
df[hierarchy_cols].head(15)

,label,level,abstract,parent_concept,parent_abstract_concept
0,Net sales,4,False,us-gaap_GrossProfit,us-gaap_StatementLineItems
1,Products,4,False,us-gaap_GrossProfit,us-gaap_StatementLineItems
2,iPhone,4,False,us-gaap_GrossProfit,us-gaap_StatementLineItems
3,Mac,4,False,us-gaap_GrossProfit,us-gaap_StatementLineItems
4,iPad,4,False,us-gaap_GrossProfit,us-gaap_StatementLineItems
5,"Wearables, Home and Accessories",4,False,us-gaap_GrossProfit,us-gaap_StatementLineItems
6,Services,4,False,us-gaap_GrossProfit,us-gaap_StatementLineItems
7,Operating segments - Americas,4,False,us-gaap_GrossProfit,us-gaap_StatementLineItems
8,Operating segments - Europe,4,False,us-gaap_GrossProfit,us-gaap_StatementLineItems
9,Operating segments - Greater China,4,False,us-gaap_GrossProfit,us-gaap_StatementLineItems


## Visualize the Tree Structure

Use the `level` column to display the statement as an indented tree:

In [ ]:
# Build a simple indented tree view from the hierarchy
date_cols = [c for c in df.columns if c.startswith('20')]
latest_period = date_cols[0] if date_cols else None

for _, row in df.iterrows():
    indent = '  ' * int(row['level'])
    label = row['label']
    
    if row['abstract']:
        print(f"{indent}{label}:")
    elif latest_period:
        value = row[latest_period]
        if value and value == value:  # not NaN
            v = float(value)
            val_str = f"${v / 1e6:,.0f}M" if abs(v) >= 1e6 else f"${v:,.0f}"
            print(f"{indent}{label:50s} {val_str}")
        else:
            print(f"{indent}{label}")

## Extract Revenue Segments Using parent_concept

The `parent_concept` column tells you which line items roll up to a parent. Use it to find all children of a concept:

In [6]:
# Find the revenue concept
revenue_rows = df[df['label'].str.contains('net sales|revenue', case=False, na=False) & ~df['abstract']]
print("Revenue-related rows:")
print(revenue_rows[['label', 'concept', 'level']].to_string(index=False))
print()

# Find all rows whose parent_concept matches the total revenue concept
total_revenue_concept = revenue_rows.iloc[-1]['concept'] if len(revenue_rows) > 0 else None

if total_revenue_concept:
    segments = df[df['parent_concept'] == total_revenue_concept]
    print(f"\nChildren of '{total_revenue_concept}':")
    if latest_period:
        print(segments[['label', latest_period]].to_string(index=False))

Revenue-related rows:
    label                                                     concept  level
Net sales us-gaap_RevenueFromContractWithCustomerExcludingAssessedTax      4


Children of 'us-gaap_RevenueFromContractWithCustomerExcludingAssessedTax':
Empty DataFrame
Columns: [label, 2025-09-27]
Index: []


## Detailed View: All Dimensional Breakdowns

The `view` parameter controls how much data appears:
- `"standard"` — face presentation (what you see in the SEC Viewer)
- `"detailed"` — includes all dimensional breakdowns (segments, geography, etc.)
- `"summary"` — non-dimensional totals only

In [7]:
# Compare row counts across views
for view in ['summary', 'standard', 'detailed']:
    view_df = income.to_dataframe(view=view)
    print(f"{view:10s} view: {len(view_df):>3} rows")

summary    view:  18 rows
standard   view:  22 rows
detailed   view:  47 rows


In [8]:
# Detailed view shows segment breakdowns
df_detailed = income.to_dataframe(view="detailed")

# Filter to dimensional rows (segment data)
dimensional = df_detailed[df_detailed['dimension'].notna()]
print(f"Dimensional rows: {len(dimensional)}")

if not dimensional.empty and latest_period:
    print(dimensional[['label', 'dimension', latest_period]].head(10).to_string(index=False))

Dimensional rows: 47
                             label  dimension   2025-09-27
                         Net sales      False 4.161610e+11
                          Products       True 3.070030e+11
                            iPhone       True 2.095860e+11
                               Mac       True 3.370800e+10
                              iPad       True 2.802300e+10
   Wearables, Home and Accessories       True 3.568600e+10
                          Services       True 1.091580e+11
     Operating segments - Americas       True 1.783530e+11
       Operating segments - Europe       True 1.110320e+11
Operating segments - Greater China       True 6.437700e+10


## Example 2: Tesla Revenue Segment Hierarchy

Tesla's income statement has a rich multi-level hierarchy: total revenues split into segments (Automotive, Energy, Services), with Automotive further broken down into sub-categories (sales, leasing, regulatory credits).

**Before v5.21.1**, all dimensional members appeared at the same level — you couldn't tell that "Automotive sales" was a child of "Automotive Revenues."

**After v5.21.1**, the `level` column reflects the definition linkbase hierarchy, so sub-members are properly nested under their parents.

In [9]:
tsla = Company("TSLA")
tsla_filing = tsla.get_filings(form="10-K").latest()
tsla_xbrl = tsla_filing.xbrl()

# Get income statement with all dimensional breakdowns
tsla_income = tsla_xbrl.statements.income_statement()
tsla_df = tsla_income.to_dataframe(view="detailed")

# Show the revenue-related rows with their hierarchy levels
revenue_mask = tsla_df['label'].str.contains('revenue|sales|leasing|credits|energy|services', case=False, na=False)
revenue_rows = tsla_df[revenue_mask]

date_cols = [c for c in tsla_df.columns if c.startswith('20')]
tsla_latest = date_cols[0] if date_cols else None

print("Tesla Revenue Hierarchy (from detailed view):\n")
print(revenue_rows[['label', 'level', 'abstract', tsla_latest]].to_string(index=False))

Tesla Revenue Hierarchy (from detailed view):

                                 label  level  abstract   2025-12-31
                              Revenues      3      True          NaN
                              Revenues      5     False 9.482700e+10
                   Automotive Revenues      5     False 6.952600e+10
                      Automotive sales      6     False 6.582100e+10
         Automotive regulatory credits      6     False 1.993000e+09
                    Automotive leasing      6     False 1.712000e+09
         Energy generation and storage      5     False 1.277100e+10
Total revenues from sales and services      5     False 9.261400e+10
   Energy generation and storage sales      6     False 1.227000e+10
                    Services and other      6     False 1.253000e+10
 Energy generation and storage leasing      5     False 5.010000e+08
                      Cost of revenues      3      True          NaN
                Total cost of revenues      5     False 

### Build a Revenue Segment Tree

The `level` column does the heavy lifting — a short loop is all you need to turn it into a tree:

In [ ]:
# Use the level column to print Tesla's revenue segments as an indented tree
date_cols = [c for c in tsla_df.columns if c.startswith('20')]
period = date_cols[0]

for _, row in tsla_df.iterrows():
    level = int(row['level'])
    label = row['label']
    indent = '  ' * level

    if row['abstract']:
        print(f"{indent}{label.upper()}")
    else:
        value = row[period]
        if value == value and value is not None:  # not NaN
            v = float(value)
            val_str = f"${v / 1e9:.3f}B" if abs(v) >= 1e9 else f"${v / 1e6:,.0f}M"
            print(f"{indent}{label:45s} {val_str}")
        else:
            print(f"{indent}{label}")

Notice that **Automotive sales**, **Automotive regulatory credits**, and **Automotive leasing** are indented one level deeper than **Automotive Revenues** — they're children, not siblings. That nesting comes directly from the `level` column, which reflects the XBRL definition linkbase hierarchy (v5.21.1+).

The same `'  ' * level` pattern works for any company:

In [ ]:
# Apple — same pattern, same loop
for _, row in df_detailed.iterrows():
    level = int(row['level'])
    label = row['label']
    indent = '  ' * level

    if row['abstract']:
        print(f"{indent}{label.upper()}")
    else:
        value = row[latest_period]
        if value == value and value is not None:
            v = float(value)
            val_str = f"${v / 1e6:,.0f}M"
            print(f"{indent}{label:45s} {val_str}")

## Example 3: Microsoft Business Segments

Microsoft reports revenue by product (Office, Azure, LinkedIn, Gaming, etc.). The detailed view includes all segment data:

In [11]:
msft = Company("MSFT")
msft_filing = msft.get_filings(form="10-K").latest()
msft_xbrl = msft_filing.xbrl()

msft_income = msft_xbrl.statements.income_statement()
msft_df = msft_income.to_dataframe(view="detailed")

date_cols = [c for c in msft_df.columns if c.startswith('20')]
latest = date_cols[0] if date_cols else None

# Show the revenue section with hierarchy
revenue_section = msft_df[msft_df['level'] <= 3].head(20)
for _, row in revenue_section.iterrows():
    indent = '  ' * int(row['level'])
    label = row['label']
    if row['abstract']:
        print(f"{indent}{label}:")
    elif latest:
        value = row[latest]
        if value and value == value:
            print(f"{indent}{label:50s} ${value:>15,.0f}")

      Gross margin                                       $193,893,000,000
      Research and development                           $ 32,488,000,000
      Sales and marketing                                $ 25,654,000,000
      General and administrative                         $  7,223,000,000
      Income before income taxes                         $123,627,000,000
      Provision for income taxes                         $ 21,795,000,000
      Net income                                         $101,832,000,000
      Earnings per share::
      Weighted average shares outstanding::


## Derive Parent from the Level Column

The API doesn't include an explicit `parent` column yet, but the `level` column + row ordering makes it easy to derive one — just track the last label seen at each level:

In [ ]:
import pandas as pd

# Derive a parent column from the level column using a stack
parent_stack = {}  # level -> label

parents = []
for _, row in tsla_df.iterrows():
    level = int(row['level'])
    parent_stack[level] = row['label']
    parents.append(parent_stack.get(level - 1))

tsla_df = tsla_df.copy()
tsla_df['parent'] = parents

# Show revenue rows with their derived parent
revenue_mask = tsla_df['label'].str.contains('Automotive|Energy|Services', case=False, na=False)
cols = ['label', 'level', 'parent', tsla_latest]
tsla_df[revenue_mask][cols].head(10)

## When to Use Statement vs Query

EdgarTools has two ways to access XBRL data. Use the right one for your task:

| Goal | Use this | Why |
|------|----------|-----|
| Revenue segment tree | `statement.to_dataframe()` | Has `level`, `parent_concept` hierarchy columns |
| Specific fact lookup | `xbrl.query()` | Fast, filtered access to individual facts |
| Dimensional breakdowns | `statement.to_dataframe(view="detailed")` | Includes all segment/geography data |
| Cross-period comparison | `statement.to_dataframe()` | Multiple period columns side by side |
| Custom fact filtering | `xbrl.facts.query().by_dimension(...)` | Full control over fact selection |

**Key insight:** `xbrl.query()` returns flat facts -- no hierarchy. `Statement.to_dataframe()` preserves the presentation tree.

## Quick Reference

```python
from edgar import *
set_identity("your.name@example.com")

# Get a statement with hierarchy
income = Company("AAPL").get_filings(form="10-K").latest().xbrl().statements.income_statement()
df = income.to_dataframe()

# Hierarchy columns
df['level']                      # Nesting depth (0, 1, 2, ...)
df['abstract']                   # True = section header
df['parent_concept']             # Calculation parent (math rollup)
df['parent_abstract_concept']    # Presentation parent (display hierarchy)

# Control detail level
df = income.to_dataframe(view="standard")   # Face presentation
df = income.to_dataframe(view="detailed")   # All segments
df = income.to_dataframe(view="summary")    # Totals only

# Find children of a concept
children = df[df['parent_concept'] == 'us-gaap_Revenue']
```

## What's Next

- [Parse XBRL Financial Data](https://colab.research.google.com/github/dgunning/edgartools/blob/main/notebooks/xbrl-financial-data-python.ipynb) -- Browse all XBRL statements and disclosures
- [Extract Revenue and Earnings](https://colab.research.google.com/github/dgunning/edgartools/blob/main/notebooks/extract-revenue-earnings-python.ipynb) -- Pull key financial metrics
- [Compare Company Financials](https://colab.research.google.com/github/dgunning/edgartools/blob/main/notebooks/compare-company-financials-python.ipynb) -- Cross-company analysis
- [Dimension Handling Guide](https://edgartools.readthedocs.io/en/latest/xbrl/concepts/dimension-handling/) -- Deep dive on dimensional data

**Resources:**
- [EdgarTools Documentation](https://edgartools.readthedocs.io/)
- [GitHub Repository](https://github.com/dgunning/edgartools)
- [PyPI Package](https://pypi.org/project/edgartools/)

---

## Support EdgarTools

If you found this tutorial helpful:

- **Star the repo** -- [github.com/dgunning/edgartools](https://github.com/dgunning/edgartools)
- **Visit edgartools.io** -- [edgartools.io](https://www.edgartools.io/)
- **Report issues** -- [Open an issue](https://github.com/dgunning/edgartools/issues)

*edgartools is free, open-source, and community-driven. No API key or paid subscription required.*